In [1]:
%config InlineBackend.figure_format = 'retina'

import os
from math import sqrt
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy import units as u
from astropy.time import Time
from astropy.io import fits

INFO:numexpr.utils:Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
INFO:numexpr.utils:Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO:numexpr.utils:NumExpr defaulting to 16 threads.


In [2]:
# Read in WAAS position data downloaded from the FAA web page

file_260106 = '/sdf/home/c/cwalter/notebooks/geosynchronous-satellites/WAAS-files/waas_output_260106.csv'
file_260303 = '/sdf/home/c/cwalter/notebooks/geosynchronous-satellites/WAAS-files/waas_output_260303.csv'

waas_260303 = pd.read_csv(file_260303, sep=r'\s+', header=None,
                        names=['jd','t0','ura','x','y','z','vx','vy','vz','ax','ay','az','f0','f1'])

waas = pd.concat([waas_260303])

jd_time = Time(waas.jd, format='jd')
waas['date'] = pd.to_datetime(jd_time.utc.isot, utc=True)
waas.set_index('date', inplace=True)

waas = waas[['x','y','z']]
waas.head(5)

,x,y,z
date,,,
2026-03-03 00:00:47+00:00,-19134144.32,-37575177.76,-4144.4
2026-03-03 00:05:03+00:00,-19134231.52,-37575188.00,-4132.0
2026-03-03 00:09:19+00:00,-19134320.48,-37575196.32,-4117.6
2026-03-03 00:13:35+00:00,-19134410.40,-37575202.56,-4102.4
2026-03-03 00:17:51+00:00,-19134501.60,-37575206.88,-4086.0


In [4]:
def get_interpolated_waas(df, tai_time, resample_period='30.0s'):

    stop_time = tai_time + 15.0*u.min
    start_time = tai_time - 15.0*u.min
    df_timewindow = df.loc[start_time.utc.iso:stop_time.utc.iso]
    
    df_resampled = df_timewindow.resample(resample_period).asfreq()
    combined = pd.concat([df_resampled, df_timewindow]).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')] # Why is this only necessary now?!

    combined.interpolate(method='cubicspline', inplace=True)

    return df_timewindow, combined

def calculate_waas_position(interpolated, tai_time, shutter_offset = 0.0 * u.second, time_offset=0):

    # Add a delta time to the header time if necessary.
    tai_time = tai_time + timedelta(seconds=time_offset)

    find_time = pd.to_datetime(tai_time.utc.isot, utc=True)
    found_index = interpolated.index.get_indexer([find_time], method='nearest')[0]

    found_time = interpolated.index[found_index]
    found_x = interpolated.iloc[found_index].x * u.m
    found_y =  interpolated.iloc[found_index].y * u.m
    found_z =  interpolated.iloc[found_index].z * u.m
    
    calc_time = tai_time.utc + shutter_offset  # Note the .utc not really necessary. 
    ground = EarthLocation.of_site('Rubin Observatory', refresh_cache=True)
    satellite = EarthLocation.from_geocentric(found_x, found_y, found_z)
    print(satellite)
    
    found_altaz = satellite.get_itrs(obstime=calc_time, location=ground).transform_to(AltAz(obstime=calc_time, location=ground))
    altaz_coordinate = SkyCoord(alt = found_altaz.alt, az = found_altaz.az, obstime=calc_time, location=ground, frame='altaz')
    icrs = altaz_coordinate.transform_to('icrs')
    
    print(found_altaz)
    print(altaz_coordinate)
    return found_time, icrs.ra.deg, icrs.dec.deg

tai_time = Time("2026-03-03T06:15:41.971", scale='tai') # Shutter Open - 439
#tai_time = Time("2026-03-03T06:15:56.390", scale='tai') # Shutter Open - 440
df_timewindow, interpolated = get_interpolated_waas(waas, tai_time, '.01s')
found_time, found_ra, found_dec = calculate_waas_position(interpolated, tai_time, shutter_offset= (0.9/2.0)*(7/8) * u.second)
print(tai_time.utc, found_time, found_ra, found_dec)

(-19139476.386502597, -37570687.975102305, 390.73908362153657) m
<AltAz Coordinate (obstime=2026-03-03T06:15:05.365, location=(1818939.006697467, -5208471.035307804, -3195171.415436697) m, pressure=0.0 hPa, temperature=0.0 deg_C, relative_humidity=0.0, obswl=1.0 micron): (az, alt, distance) in (deg, deg, m)
    (295.71771356, 29.09912845, 38688239.53781886)>
<SkyCoord (AltAz: obstime=2026-03-03T06:15:05.365, location=(1818939.006697467, -5208471.035307804, -3195171.415436697) m, pressure=0.0 hPa, temperature=0.0 deg_C, relative_humidity=0.0, obswl=1.0 micron): (az, alt) in deg
    (295.71771356, 29.09912845)>
2026-03-03T06:15:04.971 2026-03-03 06:15:04.970000+00:00 131.53748978791728 4.835497009028137


In [ ]:
# Get hand scanned satellite data from processed fits files

satellite_data = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRs-6gh1dIAjqTUdN-UOpQmaohx1U3ecO8vrb2EHUbBkoPeV14XWbtPW60-vy7cQt1Wcf1022tSo61J/pub?gid=1240073687&single=true&output=csv'
satellite_df = pd.read_csv(satellite_data)
satellite_df = satellite_df.set_index(['obsid', 'satellite'])

satellite_df = satellite_df.drop('Notes', axis=1)
satellite_df = satellite_df.dropna(how='all')
satellite_df

In [ ]:
# Read metadata from processed FITS files and build a dataframe

records_list = []

for entry in sorted(os.scandir('/sdf/home/c/cwalter/test-fits'), key=lambda x: x.name):
    with fits.open(entry.path) as hdu_list: 
       
        hdu = hdu_list[0]
        obs_id = hdu.header['OBSID']+'_'+ hdu.header['RAFTBAY'] +'_'+ hdu.header['CCDSLOT']
        tai_obs_date = Time(hdu.header['DATE-BEG'], scale='tai')
        
        try:
            shutter_delay = satellite_df.xs(obs_id).Shutter_offset.values[0] / 1000.0 * u.second
        except KeyError:
            shutter_delay = 0.0 * u.second
        
        # Get WAAS Info
        df_timewindow, interpolated = get_interpolated_waas(waas, tai_obs_date, '.01s')
        waas_tuple = calculate_waas_position(interpolated, tai_obs_date, shutter_offset = shutter_delay)
        records_list.append([obs_id] + [tai_obs_date] + [*waas_tuple])

df = pd.DataFrame(records_list, columns=['obsid', 'tai_time', 'waas_time', 'waas_RA', 'waas_DEC'])
df = df.merge(satellite_df, how='inner', on=['obsid'])

df[['obsid', 'tai_time', 'waas_time', 'waas_RA', 'waas_DEC','RA_Start', 'DEC_Start']]

In [ ]:
EUTEL_WAAS = df.assign(delta_ra = lambda x: (x.RA_Start - x.waas_RA)*3600) \
       .assign(delta_dec = lambda x: (x.DEC_Start - x.waas_DEC)*3600) \
       .assign(abs_delta_ra = lambda x: abs(x.delta_ra)) \
       .assign(abs_delta_dec = lambda x: abs(x.delta_dec)) 

#EUTEL_WAAS[['tai_time', 'delta_ra', 'Error_Start']]

In [ ]:
#EUTEL_WAAS.plot.scatter('delta_ra', 'delta_dec', marker='o', s=50)
plt.errorbar(EUTEL_WAAS.delta_ra, EUTEL_WAAS.delta_dec, fmt='o', ms=7.07, xerr=EUTEL_WAAS.Error_Start, elinewidth=0.2)

plt.title('Orbital offsets from Calculation')
plt.xlabel('$\Delta RA$ in arcsec')
plt.ylabel('$\Delta DEC$ in arcsec')
#plt.xlim(-0.13851634332084473, 2.0573575799313915);
plt.xlim(-1.0, 1.5)
print(f'Delta Dec mean is {EUTEL_WAAS.delta_dec.mean():4.3f}')

In [ ]:
# Look at the distribution of delta_ras

EUTEL_WAAS.delta_ra.plot.hist(bins=6, histtype='step', label='All Observations')

plt.xlabel('$\Delta RA$ in arcsec')
plt.legend(fontsize=10);

print(f'STD of delta_RA is {EUTEL_WAAS.delta_ra.std():4.3f}')

In [ ]:
# Calculate the mean errors and the start error positions I measured
print(f'EUTEL: \t {EUTEL_WAAS.delta_ra.mean():4.2f} (+-{EUTEL_WAAS.delta_ra.std():4.2f}) +- {EUTEL_WAAS.Error_Start.mean():4.2f}')

In [ ]:
# Calculate the average error in RA but weight the entires by my measured errors 
# on the start positions.
#
# See https://stackoverflow.com/questions/2413522/weighted-standard-deviation-in-numpy

values = EUTEL_WAAS.delta_ra
weights = EUTEL_WAAS.Error_Start
weighted_avg = np.average(values, weights=weights)
weighted_std = sqrt(np.average((values-weighted_avg)**2, weights=weights))

print('Rough guess on precision:')
print(f'Weighted averages: {weighted_avg:4.2f}" +- {weighted_std:4.2f}"')
print(f'In milliseconds: {weighted_avg/15*1000:4.2f} ms +- {weighted_std/15*1000:4.2f} ms')
print(f'In milliseconds scaled by sqrt(10): {weighted_avg/15*1000:4.2f} ms +- {weighted_std/15*1000/sqrt(10):4.2f} ms')

In [ ]:
# Functions for fit
# Needs WAAS external global data frame defined:

def calculate_position_df(data_frame, tai_vector, time_offset):  
    
    sat_ra = np.zeros(len(tai_vector))
    sat_dec = np.zeros(len(tai_vector))
    
    # For each satellite and TLE find the position
    data_frame = data_frame.reset_index()  # make sure indexes pair with number of rows
    for i, row in data_frame.iterrows():
        
        # Get WAAS Info
        df_timewindow, interpolated = get_interpolated_waas(waas, tai_vector[i], '.01s')
        
        shutter_offset = df.Shutter_offset.values[i] / 1000.0 * u.second

        # Deal with the shutter calculation (see timing notebook)

        # X, +
        # shutter_timing = [168.5890626301015,
        #     239.9243540314839,
        #     143.4781668293766,
        #     264.96699166176955,
        #     208.75905973751674,
        #     174.74614494558105,
        #     149.62742138353204,
        #     258.83681136228785,
        #     226.9164530760155,
        #     158.41229202250992]

        #Y, +
        # shutter_timing = [146.2248678111138,
        #     265.29801406998564,
        #     115.01873305691713,
        #     296.41457655210667,
        #     226.59626044723126,
        #     153.8360684497755,
        #     122.64573090992913,
        #     288.8618028801728,
        #     256.48992736423554,
        #     126.19504951398113]

        #X, - ****
        shutter_timing = [227.41093736989848,
            156.07564596851606,
            252.5218331706234,
            131.03300833823042,
            187.24094026248326,
            221.25385505441892,
            246.37257861646796,
            137.1631886377122,
            169.08354692398453,
            237.58770797749008]

        # #Y,-
        # shutter_timing = [249.7751321888862,
        #     130.7019859300144,
        #     280.98126694308286,
        #     99.58542344789332,
        #     169.40373955276877,
        #     242.16393155022453,
        #     273.3542690900709,
        #     107.13819711982725,
        #     139.51007263576446,
        #     269.80495048601887]

        #print(i, shutter_offset, shutter_timing[i])
        #shutter_offset += 200.0/ 1000.0 * u.second
        shutter_offset += shutter_timing[i]/ 1000.0 * u.second

        #waas_tuple = calculate_waas_position(interpolated, tai_vector[i], time_offset=time_offset)
        waas_tuple = calculate_waas_position(interpolated, tai_vector[i], shutter_offset=shutter_offset, time_offset=time_offset)
        sat_ra[i], sat_dec[i] = waas_tuple[1], waas_tuple[2]

    return sat_ra, sat_dec

def pdf(modified_julian_dates, time_offset=0):
    # Turn the MJD float back into a TAI Time item (seems to be good to 1e-12 seconds)
    tai_vector = [Time(item, format='mjd', scale='tai') for item in modified_julian_dates]
    sat_ra, sat_dec =  calculate_position_df(df, tai_vector, time_offset)
    
    return sat_ra

def pdf_dec(modified_julian_dates, time_offset=0):
    # Turn the MJD float back into a TAI Time item (seems to be good to 1e-12 seconds)
    tai_vector = [Time(item, format='mjd', scale='tai') for item in modified_julian_dates]
    sat_ra, sat_dec =  calculate_position_df(df, tai_vector, time_offset)
    
    return sat_dec

In [ ]:
# Test PDF function
# The PDF function requires a float.  So turn the TAI time into MJD
# mjds = [item.mjd for item in df.tai_time]
# df['calc_ra'] = pdf(mjds, time_offset = 0.0)
# df[['obsid','waas_RA', 'waas_DEC','RA_Start', 'DEC_Start', 'calc_ra']]
#df.Shutter_offset.values / 1000.0 * u.second

In [ ]:
from iminuit import Minuit
from iminuit.cost import LeastSquares, NormalConstraint

def viz(args):
    plt.hist((measured_ra - pdf(times, 0))*3600, bins=6, histtype='step', label='System Time')
    plt.hist((measured_ra - pdf(times, *args))*3600, bins=6, histtype='step', label='Fitted Offset')

    plt.legend(loc="upper right")
    plt.xlabel('$\Delta RA$ in arcsec')

times = [item.mjd for item in df.tai_time]
measured_ra = df.RA_Start.values
error_ra = df.Error_Start.values/3600

least_squares = LeastSquares(times, measured_ra, error_ra, pdf) 
least_squares.visualize = viz
least_squares.verbose = 0

m = Minuit(least_squares, time_offset=0.0)  # starting values for offset
m.fixed['time_offset'] = False

m.simplex()
m.migrad()  # finds minimum of least_squares function
m.hesse()  # accurately computes uncertainties

In [ ]:
# Manual Check of fit quality

mjds = [item.mjd for item in df.tai_time]
new_error = df.Error_Start.values/3600

chi2 = (((measured_ra - pdf(mjds, time_offset = m.values[0]))/new_error)**2).sum()
print(f'Chi2 = {chi2:4.2f}')

pulls = least_squares.pulls(m.values)
print(f'Pulls mean = {pulls.mean():4.2f} std = {pulls.std():4.2f}')

plt.hist((measured_ra - pdf(mjds, time_offset = m.values[0]))*3600, histtype='step')
plt.vlines(-0.2, 0, 3.2, linestyles='dotted', color='r')
plt.vlines(0.2, 0, 3.2, linestyles='dotted', color='r')
plt.xlabel('$\Delta RA$ in arcsec')

plt.xlim(-.5, .5)
plt.ylim(0, 3.2)


In [ ]:
print(f'{m.values[0]*1000:5.2f} +- {m.errors[0]*1000:4.2f} milliseconds')
print(f'{m.values[0]*1000*.075:5.2f} +- {m.errors[0]*1000*.075:4.2f} pixels')
print(f'FCN: {m.fcn(m.values):4.4f}')
print()
print(f'{m.values[0]/m.errors[0]:2.2} sigma from zero')

m.params

In [ ]:
m.draw_mnprofile("time_offset");

In [ ]:
shutter_aperture_size = 236 # mm square
focal_plane_size = 123.38 # mm square
shutter_readout = 396 # milliseconds

shutter_delay = shutter_readout/2.0
shutter_delay_error = shutter_readout/2.0 * focal_plane_size/shutter_aperture_size


offset = m.values[0]*1000 - ( shutter_delay )
offset_error = sqrt( (m.errors[0]*1000)**2 + shutter_delay_error**2 )

print(f'{offset:5.1f} +- {offset_error:4.1f} milliseconds')
print(f'{offset/offset_error:4.2f} sigma')

In [ ]:
pd.DataFrame({'mjds': mjds, 'delta_ra':(measured_ra - pdf(mjds, time_offset = m.values[0]))*3600})

In [ ]:
measured_dec = df.DEC_Start.values
delta_dec = (measured_dec - pdf_dec(mjds, time_offset = m.values[0]))*3600
plt.hist(delta_dec, histtype='step')
print(f'Delta Dec mean is {EUTEL_WAAS.delta_dec.mean():4.3f}')

In [ ]:
table = df[['obsid', 'waas_RA', 'waas_DEC', 'RA_Start',
       'DEC_Start', 'Error_Start', 'Shutter_offset', 'Blade']]
print(table.to_latex(float_format="{:.6f}".format))

In [ ]:
table

In [ ]:
df.columns